In [1]:
from datetime import time
from os import times
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import math
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter

#math的相关函数，import后可以直接使用
from math import radians
from math import sin
from math import cos
from math import asin
from math import sqrt

# np.set_printoptions(threshold=np.inf)#显示所有的数组
from geopy.distance import great_circle
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial'
import seaborn as sns
from scipy import stats
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
import cartopy.mpl.ticker as cticker
from scipy.ndimage import gaussian_filter

In [2]:
infile = xr.open_dataset('D:/data/ibtracs/IBTrACS.WP.v04r01.nc')


# In[4]:


stayyy=1949
endyyy=2024
censea=infile['season']
indsea=np.where((censea>=stayyy)&(censea<=endyyy))
numsea=censea[indsea]


# In[5]:


tim=infile['time']
centim=tim.dt.month[:,0]
indtim=np.where((centim==7)|(centim==8)|(centim==9)|(centim==10))
numtim=tim[indtim]


# In[6]:


centyp=infile['track_type']
indtyp=np.where(centyp==b'main')
numtyp=centyp[indtyp]


# In[7]:


cenlat=infile['usa_lat']
nummia=np.sum(np.isnan(cenlat), axis=1)
numnma=np.sum(~np.isnan(cenlat), axis=1)  
minlat=np.amin(cenlat, axis=1)
minlat=np.amin(cenlat, axis=1)
indlat=np.where(numnma>=16)
numlat=cenlat[indlat]


# In[8]:


cenlon=infile['usa_lon']
nummio=np.sum(np.isnan(cenlon), axis=1) 
numnmo=np.sum(~np.isnan(cenlon), axis=1)  
minlon=np.amin(cenlon, axis=1)  
maxlon=np.amax(cenlon, axis=1) 
indlon=np.where(numnmo>=16)
numlon=cenlon[indlon]


# In[9]:


cenwin=infile['usa_wind']
nummiw=np.sum(np.isnan(cenwin), axis=1)   #每行nan个数
numnmw=np.sum(~np.isnan(cenwin), axis=1)  
maxwin = np.amax(cenwin, axis=1) 
indmaw = np.where(maxwin>=35) 
indwin = np.where((numnmw>=16)&(maxwin>=35))
numwin =cenwin[indwin]
# numwin


# In[10]:


cenatu=infile['nature']
numnat= np.sum(np.isin(cenatu, [b'DS', b'ET']), axis=1)#有多少个DS,ET,要这俩=0的地方
numb=np.sum(cenatu != b'', axis=1)#非空值的个数，用于去除空值以及小于48小时的情况,numb>16,3h
indnat=np.where((numb!=numnat)&(numb>16))


# In[11]:


lat = infile['usa_lat']

# 创建布尔掩码以筛选7、8、9月份的数据
mask = lat['time'].dt.month.isin([7, 8, 9])

# 保留掩码对应的数据并将其他数据设为NaN
lat2 = lat.where(mask, drop=True)

# 打印筛选后的数据以检查结果
# print(lat2)


# ## 登陆landindall_flat

# In[12]:


cendis=infile['dist2land']
nummid=np.sum(np.isnan(cendis), axis=1)   
numnmd=np.sum(~np.isnan(cendis), axis=1)  #非nan数
num0md=np.sum(cendis==0, axis=1) #每行几个0
inddis=np.where((numnmd>=16)&(num0md>0))  #0代表登陆，看有几个0
numdis=cendis[inddis]#登录的结果
landindall=[]
landindallin=[]
for j in range(1982,2024):  
    indall = np.where((censea==j)
                    &(centyp==b'main')
                    &((centim==7)|(centim==8)|(centim==9)|(centim==10))
                    &(numnma>=16)
                    &(numnmo>=16)
                    &((numnmw>=16)&(maxwin>=35))
                    &((numb!=numnat)&(numb>=16))
                    &((numnmd>=16)&(num0md>0)))
    landindall.append(indall[0])
landindall_flat = np.concatenate(landindall).tolist()  # 将所有数组拼接并转换为列表


In [3]:
res=[]
df = pd.read_excel(f'Typhoon_Cluster_1.xlsx')  
id = df['Typhoon ID']
for i in range(len(id)):  # 循环台风ID
    indall = id[i]
    # print(indall)
    arrland= infile['dist2land'][indall]
    arrfall=infile['landfall'][indall]
    has_landfall = np.any((arrfall == 0) & (arrland == 0))
    # print(has_landfall)
    res.append(has_landfall)

landfall_ratio = np.sum(res) / len(res)
print(f"登陆台风占比：{landfall_ratio:.2%}") 
    

登陆台风占比：16.67%


In [4]:
res=[]
df = pd.read_excel(f'Typhoon_Cluster_2.xlsx')  
id = df['Typhoon ID']
for i in range(len(id)):  # 循环台风ID
    indall = id[i]
    # print(indall)
    arrland= infile['dist2land'][indall]
    arrfall=infile['landfall'][indall]
    has_landfall = np.any((arrfall == 0)& (arrland == 0))
    # print(has_landfall)
    res.append(has_landfall)

landfall_ratio = np.sum(res) / len(res)
print(f"登陆台风占比：{landfall_ratio:.2%}") 

登陆台风占比：89.90%


In [5]:
res=[]
df = pd.read_excel(f'Typhoon_Cluster_3.xlsx')  
id = df['Typhoon ID']
for i in range(len(id)):  # 循环台风ID
    indall = id[i]
    # print(indall)
    arrland= infile['dist2land'][indall]
    arrfall=infile['landfall'][indall]
    has_landfall = np.any((arrfall == 0)& (arrland == 0))
    # print(has_landfall)
    res.append(has_landfall)

landfall_ratio = np.sum(res) / len(res)
print(f"登陆台风占比：{landfall_ratio:.2%}") 

登陆台风占比：50.85%
